# Week 3 – Day 1: Data Cleaning with Python

This in-class notebook introduces **data cleaning with Python (pandas)** using small, readable examples.

**Learning outcomes:** By the end of this session, you should be able to:
- explain why data cleaning matters for analysis and machine learning,
- spot common data quality issues in tabular data,
- handle missing values with simple strategies (drop, mean/median/mode),
- detect and remove duplicate rows,
- distinguish **noise** from **outliers** and apply simple treatments,
- apply basic smoothing to noisy signals,
- use common transformations: scaling, encoding, binning, simple feature creation,
- standardize inconsistent formats and labels,
- correct simple inaccuracies,
- validate a cleaned dataset with quick checks,
- describe a sensible, repeatable cleaning workflow.

**Today’s roadmap:**
1. Setup and a small messy dataset
2. Quick inspection
3. Cleaning workflow overview
4. Missing data → duplicates → noise/outliers → smoothing
5. Transformations (scale, encode, bin, new features)
6. Standardization and typo fixes
7. Skew / log transform
8. Validation + compact end-to-end pipeline
9. Best practices, mini exercises, recap

> Tip: run the notebook **from top to bottom** the first time.


## 1. Introduction

**What is data cleaning?**  
Data cleaning turns **raw, messy inputs** into a **reliable table** you can analyze or feed into models.

**Why it matters:**  
Bad values (missing dates, duplicate orders, inconsistent city names) create misleading KPIs, biased models, and broken joins.

**How it supports ML/analytics:**  
Most real projects spend significant time on cleaning. Clean features improve stability, comparability, and interpretability of results.


## 2. Setup

We import libraries, set plotting defaults, and fix a **random seed** so results are reproducible in class.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.impute import KNNImputer

# Reproducibility
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Repository root (notebook lives in notebooks/week3/)
REPO_ROOT = Path.cwd().resolve().parents[1]
DATA_DIR = REPO_ROOT / "data" / "week3"
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Pandas / plots
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 110)

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("ggplot")
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.titlesize"] = 12

print("REPO_ROOT:", REPO_ROOT)
print("DATA_DIR :", DATA_DIR)


**Interpretation:** We can now save/load teaching files under `data/week3/` and get stable plots across runs.


## 3. Create or load a small messy dataset

We build a **synthetic retail / orders** table (~30 rows) with intentional problems:
missing values, duplicates, messy categories, mixed date formats, outliers, and a skewed numeric column.

**Try it yourself:** After running the cell, skim one row and name *two* problems you see without code.


In [ ]:
rows = [
    # order_id, customer_id, order_date, city, category, quantity, order_amount, shipping_days, department
    ["O001", "C10", "2026-01-05", "  new york ", "electronics", 2, 49.99, 3, "HR"],
    ["O002", "C11", "01/06/2026", "NYC", "Electronics", np.nan, 120.0, 2, "hr"],
    ["O003", "C12", "2026/13/40", "Chicago", "home", 1, 15.5, np.nan, "Finance"],
    ["O004", "C10", "2026-01-05", "  new york ", "electronics", 2, 49.99, 3, "HR"],  # duplicate row
    ["O005", "C13", "06-01-2026", "New York", np.nan, 3, -5.0, 1, "finance"],  # missing category
    ["O006", "C14", "2026-01-08", "Chcago", "electronics", 0, 89.0, 4, "HR"],
    ["O007", "C15", "2026-01-09", "boston", "sports", 1, 250.0, 2, "Ops"],
    ["O008", "C16", "2026/01/10", "BOSTON", "Sports", 2, 40.0, 5, "ops"],
    ["O009", "C17", "2026-01-11", "Seattle", "electronics", np.nan, 60.0, 1, "HR"],
    ["O010", "C18", "2026-01-12", "seattle", "Electronics", 1, 55.0, 2, "Hr"],
    ["O011", "C19", "2026-01-13", "Austin", "home", 4, 22.0, 0, "Finance"],
    ["O012", "C20", "2026-01-14", "AUSTIN", "Home", 2, 30.0, 3, "finance"],
    ["O013", "C21", "2026-01-15", "Miami", "beauty", 1, 18.0, 2, "Marketing"],
    ["O014", "C22", "2026-01-16", "miami", "Beauty", np.nan, 20.0, 1, "marketing"],
    ["O015", "C23", "2026-01-17", "Denver", "electronics", 10, 9999.0, 2, "HR"],  # outlier amount
    ["O016", "C24", "2026-01-18", "denver", "electronics", 1, 70.0, 4, "hr"],
    ["O017", "C25", "2026-01-19", "Portland", "sports", 2, 45.0, 6, "Ops"],
    ["O018", "C26", "2026-01-20", "PORTLAND", "SPORTS", 1, 50.0, 2, "OPS"],
    ["O019", "C27", "2026-01-21", "Dallas", "home", 2, 33.0, np.nan, "Finance"],
    ["O020", "C28", "2026-01-22", "dallas", "HOME", 1, 28.0, 1, "finance"],
    ["O021", "C29", "2026-01-23", "Phoenix", "beauty", 3, 12.0, 2, "Marketing"],
    ["O022", "C30", "2026-01-24", "PHOENIX", "beauty", 1, 14.0, 3, "Marketing"],
    ["O023", "C31", "2026-01-25", "Atlanta", "electronics", 2, 95.0, 2, "HR"],
    ["O024", "C32", "2026-01-26", "atlanta", "electronics", 1, 110.0, 1, "hr"],
    ["O025", "C33", "2026-01-27", "Detroit", "sports", 1, 38.0, 4, "Ops"],
    ["O026", "C34", "2026-01-28", "DETROIT", "sports", 2, 42.0, 2, "ops"],
    ["O027", "C35", "2026-01-29", "Orlando", "home", np.nan, 25.0, 1, "Finance"],
    ["O028", "C36", "2026-01-30", "orlando", "home", 2, 27.0, 2, "finance"],
]

cols = [
    "order_id",
    "customer_id",
    "order_date",
    "city",
    "category",
    "quantity",
    "order_amount",
    "shipping_days",
    "department",
]

df_messy = pd.DataFrame(rows, columns=cols)

# Skewed helper column (total spend proxy) — some zeros/missing later
df_messy["line_total"] = df_messy["quantity"] * df_messy["order_amount"]
df_messy.loc[rng.choice(len(df_messy), size=4, replace=False), "line_total"] = np.nan

messy_path = DATA_DIR / "week3_messy_data.csv"
df_messy.to_csv(messy_path, index=False)
print("Saved:", messy_path)
display(df_messy.head(10))


**Interpretation:** The CSV is intentionally imperfect so we can practice realistic fixes. The same file is saved for reuse or homework.


## 4. Quick data inspection

We always start with a **quick profile**: shape, types, summaries, missing counts, duplicates, and a few `unique()` checks.


In [ ]:
df = df_messy.copy()

print("Shape:", df.shape)
display(df.head())

df.info()

display(df.describe(include="all").transpose())

print("Missing per column:\n", df.isna().sum())
print("Full-row duplicates:", df.duplicated().sum())

print("Unique cities (raw):", df["city"].nunique())
display(df["city"].unique()[:12])

print("Unique categories (raw):", df["category"].unique())


**Interpretation:** `info()` shows dtypes and non-null counts; `describe(include="all")` surfaces numeric ranges and top strings. High `unique()` counts for cities/categories often mean **inconsistent labels** rather than true diversity.


## 5. Data cleaning workflow overview

A practical loop:

1. **Profile** — understand issues (missing, duplicates, formats).
2. **Transform** — impute, standardize, encode, rescale, fix labels.
3. **Validate** — re-check constraints (ranges, uniqueness, dates).
4. **Deliver** — save a clean table + short changelog.

The sections below map to this loop: we profile first, then transform step by step, then validate at the end.


## 6. Handling missing data

**Concept:** Missing values break sums, averages, and many ML models. Strategies include **deleting** sparse rows/columns, **imputing** with typical values, or **flagging** that data was missing.

**Try it yourself:** Before imputing `quantity`, decide: would **median** or **mode (1)** be safer for this toy store? Why?


In [ ]:
# Track which cells were missing before imputation
df["quantity_was_missing"] = df["quantity"].isna()
df["shipping_was_missing"] = df["shipping_days"].isna()

before = df[["quantity", "shipping_days", "line_total"]].copy()

# Numeric: median imputation for quantity and shipping_days
for col in ["quantity", "shipping_days"]:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

# Categorical: mode imputation for category (simple, common baseline)
if df["category"].isna().any():
    mode_cat = df["category"].mode(dropna=True).iloc[0]
    df["category"] = df["category"].fillna(mode_cat)

# line_total: recompute where possible, then median for any remainder
df["line_total"] = df["quantity"] * df["order_amount"]
df["line_total"] = df["line_total"].fillna(df["line_total"].median())

print("Before (sample):")
display(before.head(8))
print("After (same rows):")
display(df.loc[before.index, ["quantity", "shipping_days", "line_total"]].head(8))

print("Remaining missing values:", df.isna().sum().sum())

# Demo: row deletion with dropna() on a fresh copy of the messy table (before imputation)
tmp = df_messy.copy()
print("\nDropna demo on messy copy:")
print("  rows before:", len(tmp))
print("  rows after dropna() (any missing in row):", len(tmp.dropna()))
print("  rows after dropna(subset=['quantity'], how='any'):", len(tmp.dropna(subset=['quantity'], how='any')))


**Interpretation:** Median imputation is **robust to outliers** in skewed numeric columns. `dropna()` removes entire rows (or columns with `axis=1`)—fast, but it can **bias** results if missingness is not random, and it **loses information**. Prefer explicit rules and documentation.


### Optional note: KNN imputation (light touch)

`KNNImputer` uses nearby rows to guess missing numerics. It is **not** required for this class demo—keep it in mind for richer numeric tables.


In [ ]:
# Tiny optional demo: 3 numeric columns, artificial missing
demo = df[["quantity", "order_amount", "shipping_days"]].copy()
demo.iloc[0, 0] = np.nan
imputer = KNNImputer(n_neighbors=3)
filled = imputer.fit_transform(demo)
print("First row after KNNImputer:", filled[0])


## 7. Removing duplicates

**Concept:** Exact duplicate rows can **double-count** revenue or inflate training data. `drop_duplicates()` keeps the first occurrence by default.

**Try it yourself:** Should we dedupe on `order_id` only, or on the full row? When might they differ?


In [ ]:
before_n = len(df)
dup_mask = df.duplicated()
print("Duplicate rows:", dup_mask.sum())
display(df.loc[dup_mask].head())

df = df.drop_duplicates(keep="first").reset_index(drop=True)
after_n = len(df)
print(f"Rows before: {before_n}, after: {after_n}, removed: {before_n - after_n}")


**Interpretation:** After deduplication, aggregates (sums, counts) better match the **true number of orders**.


## 8. Noise vs outliers

**Noise** — random small fluctuations that make values “jittery” (measurement error, typing variance). Often high frequency and not meaningful alone.

**Outlier** — a value **far from the bulk** of the distribution; may be an error *or* a rare but real event (fraud, VIP order).

**Difference (plain language):** Noise is “static on the line”; an outlier is “a point far from the cluster.” Context and domain knowledge decide how to treat each.


### 8A. Numeric outliers (IQR + boxplot)


In [ ]:
col = "order_amount"
plt.boxplot(df[col].dropna(), vert=True)
plt.title("Boxplot: order_amount")
plt.ylabel(col)
plt.show()

q1 = df[col].quantile(0.25)
q3 = df[col].quantile(0.75)
iqr = q3 - q1
low = q1 - 1.5 * iqr
high = q3 + 1.5 * iqr
outliers = df[(df[col] < low) | (df[col] > high)]
print(f"IQR fences: [{low:.2f}, {high:.2f}]")
display(outliers[["order_id", col]])


**Interpretation:** Points outside the whiskers are **candidates** for review—not automatically wrong. Here the huge `order_amount` is a classic teaching outlier.


### 8B. Noise in a time series (synthetic)

We plot a smooth signal vs the same signal plus Gaussian noise.


In [ ]:
t = np.linspace(0, 4 * np.pi, 120)
signal = np.sin(t)
noise = rng.normal(0, 0.25, size=t.shape)
noisy = signal + noise

plt.plot(t, signal, label="clean signal", linewidth=2)
plt.plot(t, noisy, label="noisy signal", alpha=0.7)
plt.title("Clean vs noisy time series (demo)")
plt.legend()
plt.show()


**Interpretation:** The noisy curve **follows** the pattern but wiggles; smoothing (next sections) reduces wiggles but can also blur real spikes.


## 9. Dealing with noise and outliers (simple tools)

We can **smooth** (rolling mean), **cap** values (winsorize), or **set suspicious values to NaN** then impute. Always record what you did for reproducibility.

**Try it yourself:** Cap `order_amount` at the 99th percentile instead of the IQR fence—when would that be preferable?


In [ ]:
# Mark extreme order_amount as missing, then fill with median (simple teaching pattern)
col = "order_amount"

# Fix obvious sign errors first (business rule: amounts should be non-negative here)
df.loc[df[col] < 0, col] = np.nan
df[col] = df[col].fillna(df[col].median())

q1 = df[col].quantile(0.25)
q3 = df[col].quantile(0.75)
iqr = q3 - q1
low = q1 - 1.5 * iqr
high = q3 + 1.5 * iqr

df["order_amount_outlier"] = (df[col] < low) | (df[col] > high)
median_amount = df.loc[~df["order_amount_outlier"], col].median()
df.loc[df["order_amount_outlier"], col] = median_amount

print("Outliers adjusted:", int(df["order_amount_outlier"].sum()))
display(df[["order_id", "order_amount", "order_amount_outlier"]].head(10))


**Interpretation:** We **documented** outliers via a flag column and replaced with a robust center. In production, you might **reject** the row or **ask the business** instead.


## 10. Mean smoothing demo

**Rolling mean** averages neighboring points—useful for noisy sequences, less ideal if you must preserve sharp true jumps.


In [ ]:
window = 5
smoothed = pd.Series(noisy).rolling(window=window, center=True).mean()

plt.plot(t, noisy, label="noisy", alpha=0.6)
plt.plot(t, smoothed, label=f"rolling mean (window={window})", linewidth=2)
plt.title("Mean smoothing demo")
plt.legend()
plt.show()


**Interpretation:** Smoothing reduces variance but **lags** sharp changes; choose `window` based on sampling rate and domain.


## 11. Data transformation (overview)

Common transforms:
- **Scaling** — put numeric features on comparable ranges.
- **Encoding** — turn categories into numeric columns for many models.
- **Binning** — group continuous values into bands (age bands, spend tiers).
- **Feature generation** — derive new columns from existing ones.


## 12. Feature scaling

**StandardScaler** — zero mean, unit variance (common for many models).  
**MinMaxScaler** — scales to a fixed range (often [0, 1]), preserves zero entries if min is 0.

**Try it yourself:** Which scaler is more sensitive to **one huge outlier** in a column?


In [ ]:
num_cols = ["quantity", "order_amount", "shipping_days"]
X = df[num_cols].copy()

display(X.describe().transpose())

std = StandardScaler()
X_std = pd.DataFrame(std.fit_transform(X), columns=[c + "_std" for c in num_cols], index=X.index)

mm = MinMaxScaler()
X_mm = pd.DataFrame(mm.fit_transform(X), columns=[c + "_mm" for c in num_cols], index=X.index)

compare = pd.concat([X.head(), X_std.head(), X_mm.head()], axis=1)
display(compare)


**Interpretation:** After standardization, columns share comparable spread; min-max puts everything on the same bounded scale for interpretation or neural nets.


## 13. Encoding categorical data

**One-hot encoding** creates binary columns per category. We use `pandas.get_dummies` for clarity.


In [ ]:
dept_clean = df["department"].astype(str).str.strip().str.title()
dummies = pd.get_dummies(dept_clean, prefix="dept")
display(dummies.head())

df = pd.concat([df, dummies], axis=1)
print("New columns (sample):", list(dummies.columns)[:6], "...")


**Interpretation:** Each row becomes a sparse binary pattern across departments—many algorithms expect numeric input in this form.


## 14. Binning

**Binning** maps numbers to ordered categories (e.g., low / medium / high spend tiers) for reporting or tree-friendly features.


In [ ]:
df["spend_band"] = pd.cut(
    df["order_amount"],
    bins=[-np.inf, 30, 80, np.inf],
    labels=["low", "medium", "high"],
)
display(df[["order_id", "order_amount", "spend_band"]].head(10))
print(df["spend_band"].value_counts())


**Interpretation:** Bins simplify dashboards; boundaries should reflect **business rules**, not only quantiles.


## 15. Feature generation

New columns can capture **economics** or **ratios** that models find useful.


In [ ]:
df["spend_per_unit"] = df["order_amount"] / df["quantity"].replace(0, np.nan)
df["spend_per_unit"] = df["spend_per_unit"].fillna(df["spend_per_unit"].median())

display(df[["order_id", "order_amount", "quantity", "spend_per_unit"]].head(8))


**Interpretation:** `spend_per_unit` approximates **average price per item ordered**; we guarded division by zero.


## 16. Data standardization (formats and consistency)

We fix **dates** (ISO), **whitespace**, **text case**, and **synonym cities**.


In [ ]:
# Parse mixed date formats, coerce invalid (e.g. month 13) to NaT
parsed = pd.to_datetime(df["order_date"], errors="coerce", format="mixed")
# Fill rare parse failures with a fixed teaching fallback
fallback = pd.Timestamp("2026-01-01")
df["order_date_clean"] = parsed.fillna(fallback).dt.strftime("%Y-%m-%d")

# Text cleanup
df["city"] = df["city"].astype(str).str.strip().str.title()

city_map = {
    "Nyc": "New York",
    "New York": "New York",
    "New york": "New York",
    "Chcago": "Chicago",
    "Bostom": "Boston",
}
# apply canonical names where we know typos/synonyms
df["city"] = df["city"].replace(city_map)

df["category"] = df["category"].astype(str).str.strip().str.lower()

print("Unique cities after cleanup:", sorted(df["city"].unique())[:10], "...")
print("Unique categories:", sorted(df["category"].unique()))


**Interpretation:** Consistent strings make **groupby** counts trustworthy; ISO dates make sorting and joins reliable.


## 17. Correcting inaccuracies

Simple **mapping dictionaries** fix systematic typos in categorical fields.


In [ ]:
# Introduce a state column with typos for demo (synthetic column)
state_raw = np.array(
    ["CA", "ca", "NY", "Texass", "tx", "FL", "fl", "WA", "Ny", "CA"] * (len(df) // 10 + 1)
)[: len(df)]

fix_map = {
    "ca": "CA",
    "ny": "NY",
    "Ny": "NY",
    "TX": "TX",
    "tx": "TX",
    "Texass": "TX",
    "fl": "FL",
    "WA": "WA",
}

df["state_raw"] = state_raw
print("Before:", np.unique(df["state_raw"])[:12])

df["state_clean"] = df["state_raw"].replace(fix_map)
print("After:", np.unique(df["state_clean"])[:12])


**Interpretation:** `replace` with a dictionary is a fast first pass; maintain the map in code or a **reference table** for auditability.


## 18. Normalization and transformation (skew)

**Log transform** (`log1p`) compresses heavy right tails, helping visualization and some models. Compare histograms before/after.


In [ ]:
skewed = rng.lognormal(mean=1.5, sigma=0.9, size=500)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(skewed, bins=30, color="steelblue", edgecolor="white")
axes[0].set_title("Before: skewed values")

log_skewed = np.log1p(skewed)
axes[1].hist(log_skewed, bins=30, color="darkorange", edgecolor="white")
axes[1].set_title("After: log1p transform")

plt.tight_layout()
plt.show()


**Interpretation:** The right panel looks more **bell-like**; use logs only when values are positive and interpretation still makes sense.


## 19. Validation checks after cleaning

Quick **assertions** and summaries confirm the table is fit for next steps.


In [ ]:
assert df.duplicated().sum() == 0, "Unexpected duplicate rows remain"
assert df["order_date_clean"].notna().all(), "Dates should all parse or be filled"
assert df["order_amount"].min() >= 0, "Amounts should be non-negative after fixes"
assert df[["quantity", "shipping_days"]].isna().sum().sum() == 0, "Key numerics still missing"

summary = {
    "rows": len(df),
    "missing_cells": int(df.isna().sum().sum()),
    "duplicate_rows": int(df.duplicated().sum()),
}
print("Validation summary:", summary)


**Interpretation:** Validation turns cleaning into a **checklist** you can rerun whenever the pipeline refreshes.


## 20. Mini end-to-end cleaning pipeline (compact)

We **reload** the messy CSV and apply the main steps in one sequence, then save the cleaned file.


In [ ]:
raw = pd.read_csv(DATA_DIR / "week3_messy_data.csv")

def clean_orders_pipeline(d: pd.DataFrame) -> pd.DataFrame:
    out = d.copy()
    # Missing numerics
    for col in ["quantity", "shipping_days"]:
        out[col] = out[col].fillna(out[col].median())
    out["line_total"] = out["quantity"] * out["order_amount"]
    out["line_total"] = out["line_total"].fillna(out["line_total"].median())
    if out["category"].isna().any():
        out["category"] = out["category"].fillna(out["category"].mode(dropna=True).iloc[0])
    # Dedup
    out = out.drop_duplicates(keep="first").reset_index(drop=True)
    # Non-negative amounts
    col = "order_amount"
    out.loc[out[col] < 0, col] = np.nan
    out[col] = out[col].fillna(out[col].median())
    # Outlier cap on order_amount using IQR
    q1, q3 = out[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    mask = (out[col] < low) | (out[col] > high)
    med = out.loc[~mask, col].median()
    out.loc[mask, col] = med
    # Dates + strings
    parsed = pd.to_datetime(out["order_date"], errors="coerce", format="mixed")
    out["order_date_clean"] = parsed.fillna(pd.Timestamp("2026-01-01")).dt.strftime("%Y-%m-%d")
    out["city"] = out["city"].astype(str).str.strip().str.title().replace(
        {"Nyc": "New York", "Chcago": "Chicago"}
    )
    out["category"] = out["category"].astype(str).str.strip().str.lower()
    # Simple derived feature
    out["spend_per_unit"] = out["order_amount"] / out["quantity"].replace(0, np.nan)
    out["spend_per_unit"] = out["spend_per_unit"].fillna(out["spend_per_unit"].median())
    return out


df_clean = clean_orders_pipeline(raw)
clean_path = DATA_DIR / "week3_cleaned_data.csv"
df_clean.to_csv(clean_path, index=False)
print("Saved:", clean_path)
display(df_clean.head(10))


**Interpretation:** A single function (or script) encodes the **team agreement** on how this dataset should be cleaned—rerun it when new rows arrive.


## 21. Best practices (markdown)

- **Automate** repetitive steps (functions, scripts, notebooks with clear order).
- **Document** each decision: why impute, why drop, why cap outliers.
- **Prioritize correctness** over speed—bad cleaning spreads silently.
- **Version** inputs/outputs (raw vs cleaned filenames, dates).
- **Validate** with simple checks and spot audits.
- **Reproducibility:** seeds, pinned library versions, saved pipelines.

This mirrors professional ML workflows: cleaning is part of **MLOps hygiene**, not a one-off task.


## 22. In-class mini exercises

1. Find **one more** city or category inconsistency in `week3_messy_data.csv` and add a fix to `clean_orders_pipeline`.
2. Change **imputation** for `quantity` from median to a fixed value you defend (e.g., 1)—compare how `line_total` changes.
3. Build a **new binned feature** from `shipping_days` (e.g., fast / standard / slow) using `pd.cut`.

Spend ~5 minutes each; share one insight with a partner.


### Helper: summarize data quality issues (optional utility)


In [ ]:
def summarize_quality_issues(frame: pd.DataFrame) -> pd.DataFrame:
    """Return a small table of common issue counts for teaching demos."""
    dup_rows = int(frame.duplicated().sum())
    missing = int(frame.isna().sum().sum())
    all_num = frame.select_dtypes(include=[np.number])
    outlier_cols = []
    for c in all_num.columns:
        s = frame[c].dropna()
        if len(s) < 5:
            continue
        q1, q3 = s.quantile([0.25, 0.75])
        iqr = q3 - q1
        if iqr == 0:
            continue
        low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        if ((s < low) | (s > high)).any():
            outlier_cols.append(c)
    return pd.DataFrame(
        [
            ["duplicate_rows", dup_rows],
            ["missing_cells", missing],
            ["numeric_cols_with_iqr_outliers", len(outlier_cols)],
        ],
        columns=["check", "value"],
    )


display(summarize_quality_issues(pd.read_csv(DATA_DIR / "week3_messy_data.csv")))
display(summarize_quality_issues(pd.read_csv(DATA_DIR / "week3_cleaned_data.csv")))


**Interpretation:** Use helpers like this at the start and end of a pipeline to show **before/after** improvement in a single glance.


## 23. Final recap

Today we connected **profiling → transformation → validation** using pandas and small sklearn tools. You practiced missing data, deduplication, outliers vs noise, smoothing, scaling, encoding, binning, formatting, and a compact pipeline.

**What matters most:** transparent rules, reproducible code, and validation that catches regressions.

**Next:** These cleaned features feed stronger analytics and more stable ML models—always revisit cleaning when the **data source** changes.

---

**Outputs saved (optional reuse):**
- `data/week3/week3_messy_data.csv`
- `data/week3/week3_cleaned_data.csv`
